# Логистическая регрессия и метрики классификации: Pima Indians Diabetes

**Цель работы:** построить и исследовать бинарный классификатор на медицинском датасете `Pima Indians Diabetes Database` (целевая переменная `Outcome`: `1` — есть диабет, `0` — нет диабета), сравнить модель с константным baseline, проанализировать влияние порога классификации (`0.3`, `0.5`, `0.7`) и весов классов (`class_weight='balanced'`) на матрицу ошибок и метрики `Accuracy`, `Precision`, `Recall`, `F1`, `ROC-AUC`, `PR-AUC`.

## Блок 1. Подготовка данных и константный baseline

In [ ]:
import os
import shutil
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    roc_curve,
    precision_recall_curve,
    classification_report,
)

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="muted")
RANDOM_STATE = 42

In [ ]:
# 1.1 Загрузка датасета (проверяем локальный файл или копируем из ~/Downloads/diabetes.csv)
os.makedirs("data", exist_ok=True)
local_path = os.path.join("data", "diabetes.csv")
downloads_path = os.path.expanduser("~/Downloads/diabetes.csv")

if os.path.exists(downloads_path):
    shutil.copy2(downloads_path, local_path)
    print(f"Скопирован датасет из {downloads_path} -> {local_path}")

df = pd.read_csv(local_path)
print(f"Размер датасета: {df.shape[0]} строк, {df.shape[1]} столбцов")
display(df.head())

In [ ]:
# 1.2 Типы данных и базовые статистики
df.info()
display(df.describe().T)

In [ ]:
# 1.3 Проверка явных пропусков (NaN) и скрытых пропусков (нулей в физиологических признаках)
print("Явные пропуски (NaN):")
print(df.isna().sum())

# В признаках Glucose, BloodPressure, SkinThickness, Insulin, BMI физиологический 0 невозможен —
# так в оригинальном датасете закодированы пропущенные измерения.
zero_cols = ["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"]
zero_counts = (df[zero_cols] == 0).sum()
zero_pct = (df[zero_cols] == 0).mean() * 100
zero_summary = pd.DataFrame({"Количество нулей": zero_counts, "Доля (%)": zero_pct.round(2)})
print("\nСкрытые пропуски (значения == 0 в медицинских показателях):")
display(zero_summary)

In [ ]:
# 1.4 Распределение целевой переменной Outcome и проверка дисбаланса классов
class_counts = df["Outcome"].value_counts().sort_index()
class_shares = df["Outcome"].value_counts(normalize=True).sort_index() * 100

balance_df = pd.DataFrame({
    "Класс": ["0 (Нет диабета)", "1 (Диабет)"],
    "Количество": class_counts.values,
    "Доля (%)": class_shares.round(2).values
})
display(balance_df)

fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(balance_df["Класс"], balance_df["Количество"], color=["#4c72b0", "#dd8452"], width=0.55)
for bar, pct in zip(bars, balance_df["Доля (%)"]):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 8, f"{int(bar.get_height())} ({pct}%)", ha="center", fontweight="bold")
ax.set_title("Распределение целевой переменной Outcome")
ax.set_ylabel("Количество пациентов")
ax.set_ylim(0, max(balance_df["Количество"]) * 1.15)
plt.tight_layout()
plt.show()

**Наблюдение по балансу классов:**
* Класс `0` (здоровые пациенты) составляет около `65.1%` (`500` наблюдений), а класс `1` (диабет) — `34.9%` (`268` наблюдений).
* Присутствует умеренный дисбаланс классов (~`2 : 1`). Поэтому при разбиении на `train` и `test` обязательно используем стратификацию (`stratify=y`), а при оценке модели нельзя опираться только на `Accuracy`: константное предсказание `0` уже даст `Accuracy = 65.1%`, не выявив ни одного больного пациента.

In [ ]:
# 1.5 Разделение на обучающую и тестовую выборки со стратификацией
X = df.drop(columns=["Outcome"]).copy()
y = df["Outcome"].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

print(f"Train shape: {X_train.shape}, доля класса 1: {y_train.mean():.3f}")
print(f"Test shape:  {X_test.shape}, доля класса 1: {y_test.mean():.3f}")

# Заменяем скрытые нули на NaN и заполняем медианой, рассчитанной СТРОГО по train (без утечки данных)
X_train[zero_cols] = X_train[zero_cols].replace(0, np.nan)
X_test[zero_cols] = X_test[zero_cols].replace(0, np.nan)

imputer = SimpleImputer(strategy="median")
scaler = StandardScaler()

X_train_imp = imputer.fit_transform(X_train)
X_test_imp = imputer.transform(X_test)

X_train_scaled = scaler.fit_transform(X_train_imp)
X_test_scaled = scaler.transform(X_test_imp)

In [ ]:
# 1.6 Вспомогательная функция для расчёта метрик и построения матрицы ошибок
def evaluate_predictions(y_true, y_pred, y_proba=None, model_name="Model"):
    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()
    res = {
        "Модель / Порог": model_name,
        "Accuracy": round(accuracy_score(y_true, y_pred), 4),
        "Precision": round(precision_score(y_true, y_pred, zero_division=0), 4),
        "Recall": round(recall_score(y_true, y_pred, zero_division=0), 4),
        "F1": round(f1_score(y_true, y_pred, zero_division=0), 4),
        "ROC-AUC": round(roc_auc_score(y_true, y_proba), 4) if y_proba is not None else np.nan,
        "PR-AUC": round(average_precision_score(y_true, y_proba), 4) if y_proba is not None else np.nan,
        "TP": int(tp),
        "FP": int(fp),
        "FN": int(fn),
        "TN": int(tn),
    }
    return res, cm

# Константный baseline (всегда предсказывает самый частый класс 0)
dummy = DummyClassifier(strategy="most_frequent", random_state=RANDOM_STATE)
dummy.fit(X_train_scaled, y_train)
y_pred_dummy = dummy.predict(X_test_scaled)
y_proba_dummy = dummy.predict_proba(X_test_scaled)[:, 1]

baseline_metrics, baseline_cm = evaluate_predictions(
    y_test, y_pred_dummy, y_proba_dummy, model_name="Baseline (most_frequent -> 0)"
)
display(pd.DataFrame([baseline_metrics]))

**Вывод по baseline:**
Константная модель, предсказывающая всем пациентам класс `0` (здоров), получает `Accuracy = 0.6494` (`64.9%`), но при этом `Precision = 0.0`, `Recall = 0.0`, `F1 = 0.0` и пропускает всех `54` больных пациентов в тесте (`FN = 54`, `TP = 0`). Это наглядно доказывает, почему в медицинской диагностике `Accuracy` в отрыве от `Recall` и `F1` вводит в заблуждение.

---
## Блок 2. Логистическая регрессия (стандартный порог 0.5)

In [ ]:
# 2.1 Обучение логистической регрессии и проверка связи линейного скора z и вероятности p = sigmoid(z)
logreg = LogisticRegression(random_state=RANDOM_STATE, max_iter=1000)
logreg.fit(X_train_scaled, y_train)

# Предсказанные классы (по умолчанию порог 0.5) и вероятности класса 1
y_pred_05 = logreg.predict(X_test_scaled)
y_proba = logreg.predict_proba(X_test_scaled)[:, 1]

# Проверим вручную формулу сигмоиды: p = 1 / (1 + exp(-z))
z_scores = logreg.decision_function(X_test_scaled)
manual_sigmoid = 1.0 / (1.0 + np.exp(-z_scores))
print("Максимальное расхождение между predict_proba и ручной сигмоидой:", np.max(np.abs(y_proba - manual_sigmoid)))

# Таблица первых 5 пациентов: линейный скор z, вероятность p и итоговый класс
sample_preds = pd.DataFrame({
    "z = w*x + b": z_scores[:5].round(3),
    "p = predict_proba (класс 1)": y_proba[:5].round(3),
    "predict (порог 0.5)": y_pred_05[:5],
    "Истинный Outcome": y_test.values[:5]
})
display(sample_preds)

In [ ]:
# 2.2 Метрики и матрица ошибок при пороге 0.5
logreg_05_metrics, cm_05 = evaluate_predictions(
    y_test, y_pred_05, y_proba, model_name="LogisticRegression (threshold=0.5)"
)
display(pd.DataFrame([baseline_metrics, logreg_05_metrics]))

fig, ax = plt.subplots(figsize=(5.5, 4.5))
sns.heatmap(
    cm_05, annot=True, fmt="d", cmap="Blues", cbar=False,
    xticklabels=["Предсказан 0 (Здоров)", "Предсказан 1 (Диабет)"],
    yticklabels=["Истинный 0 (Здоров)", "Истинный 1 (Диабет)"],
    ax=ax
)
ax.set_title("Матрица ошибок LogisticRegression (порог = 0.5)")
plt.tight_layout()
plt.show()

In [ ]:
# 2.3 Коэффициенты модели (влияние признаков на логарифм шансов диабета)
coef_df = pd.DataFrame({
    "Признак": X.columns,
    "Коэффициент w": logreg.coef_[0].round(4),
    "Odds Ratio (exp(w))": np.exp(logreg.coef_[0]).round(4)
}).sort_values(by="Коэффициент w", ascending=False)
display(coef_df)

fig, ax = plt.subplots(figsize=(7, 4))
sns.barplot(data=coef_df, x="Коэффициент w", y="Признак", palette="vlag", ax=ax)
ax.axvline(0, color="black", linewidth=1)
ax.set_title("Коэффициенты логистической регрессии (стандартизированные признаки)")
plt.tight_layout()
plt.show()

In [ ]:
# 2.4 Построение ROC-кривой и PR-кривой, расчёт ROC-AUC и PR-AUC
fpr, tpr, _ = roc_curve(y_test, y_proba)
prec_curve, rec_curve, _ = precision_recall_curve(y_test, y_proba)

roc_auc_val = roc_auc_score(y_test, y_proba)
pr_auc_val = average_precision_score(y_test, y_proba)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))

# ROC-кривая
axes[0].plot(fpr, tpr, color="#2b7bba", lw=2.2, label=f"LogReg (ROC-AUC = {roc_auc_val:.4f})")
axes[0].plot([0, 1], [0, 1], color="gray", linestyle="--", label="Случайный классификатор (0.50)")
axes[0].set_xlabel("False Positive Rate (FPR = FP / (FP + TN))")
axes[0].set_ylabel("True Positive Rate (Recall = TP / (TP + FN))")
axes[0].set_title("ROC-кривая")
axes[0].legend(loc="lower right")

# PR-кривая
baseline_pos_rate = y_test.mean()
axes[1].plot(rec_curve, prec_curve, color="#d95f02", lw=2.2, label=f"LogReg (PR-AUC = {pr_auc_val:.4f})")
axes[1].axhline(baseline_pos_rate, color="gray", linestyle="--", label=f"Доля класса 1 ({baseline_pos_rate:.2f})")
axes[1].set_xlabel("Recall (Полнота)")
axes[1].set_ylabel("Precision (Точность)")
axes[1].set_title("Precision-Recall (PR) кривая")
axes[1].legend(loc="lower left")

plt.tight_layout()
plt.show()

---
## Блок 3. Эксперимент с порогом классификации (`0.3`, `0.5`, `0.7`)

По умолчанию `predict()` использует порог `t = 0.5`. Проверим, как меняется соотношение ошибок `FN` (пропуск болезни) и `FP` (ложная тревога) при порогах `0.3`, `0.5` и `0.7`.

In [ ]:
# 3.1 Сравнение порогов 0.3, 0.5 и 0.7 на одних и тех же вероятностях y_proba
thresholds_to_check = [0.3, 0.5, 0.7]
threshold_rows = []
cms_by_threshold = {}

for t in thresholds_to_check:
    y_pred_t = (y_proba >= t).astype(int)
    row, cm_t = evaluate_predictions(y_test, y_pred_t, y_proba, model_name=f"Порог t = {t}")
    threshold_rows.append(row)
    cms_by_threshold[t] = cm_t

threshold_comp_df = pd.DataFrame(threshold_rows)
display(threshold_comp_df)

# Визуализация 3 матриц ошибок рядом
fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
for ax, t in zip(axes, thresholds_to_check):
    sns.heatmap(
        cms_by_threshold[t], annot=True, fmt="d", cmap="YlGnBu", cbar=False,
        xticklabels=["Пред. 0", "Пред. 1"],
        yticklabels=["Ист. 0", "Ист. 1"],
        ax=ax
    )
    tn, fp, fn, tp = cms_by_threshold[t].ravel()
    ax.set_title(f"Порог t = {t}\nFN (пропуск) = {fn}, FP (ложн.) = {fp}")
plt.tight_layout()
plt.show()

In [ ]:
# 3.2 График зависимости Precision, Recall и F1 от порога классификации (0.05 .. 0.95)
grid_t = np.linspace(0.05, 0.90, 86)
prec_list, rec_list, f1_list, acc_list = [], [], [], []

for t in grid_t:
    pred_t = (y_proba >= t).astype(int)
    prec_list.append(precision_score(y_test, pred_t, zero_division=0))
    rec_list.append(recall_score(y_test, pred_t, zero_division=0))
    f1_list.append(f1_score(y_test, pred_t, zero_division=0))
    acc_list.append(accuracy_score(y_test, pred_t))

best_f1_idx = int(np.argmax(f1_list))
best_t_f1 = grid_t[best_f1_idx]

fig, ax = plt.subplots(figsize=(9, 4.8))
ax.plot(grid_t, prec_list, label="Precision (Точность)", color="#2b7bba", lw=2)
ax.plot(grid_t, rec_list, label="Recall (Полнота)", color="#d95f02", lw=2)
ax.plot(grid_t, f1_list, label="F1-мера", color="#1b9e77", lw=2.5)

for t_mark, col in [(0.3, "#e7298a"), (0.5, "#666666"), (0.7, "#7570b3")]:
    ax.axvline(t_mark, color=col, linestyle="--", alpha=0.8, label=f"t = {t_mark}")

ax.set_xlabel("Порог классификации (t)")
ax.set_ylabel("Значение метрики")
ax.set_title(f"Зависимость Precision, Recall и F1 от порога (максимум F1 при t ≈ {best_t_f1:.2f})")
ax.legend(loc="best")
plt.tight_layout()
plt.show()

**Обоснование выбора порога для медицинской задачи (скрининг диабета):**
1. **При пороге `t = 0.7`** модель становится предельно осторожной: `Precision` высокий, ложных срабатываний (`FP`) мало, но полнота (`Recall`) резко падает — больше половины больных диабетом получают ответ «здоров» (`FN` максимален). Для медицины такой порог неприемлем.
2. **При пороге `t = 0.5`** часть реальных случаев диабета всё ещё пропускается (`FN` остаётся существенным), так как положительный класс в меньшинстве (`34.9%`).
3. **При снижении порога до `t = 0.30–0.35`** модель находит значительно больше реальных случаев диабета (`Recall` вырастает до ~`0.80+`, число пропущенных больных `FN` сокращается почти вдвое), при этом `F1` достигает максимума или остаётся вблизи него. Цена такого решения — рост ложноположительных срабатываний (`FP`), но в первичном медицинском скрининге отправить пациента на повторный лабораторный анализ крови (`FP`) намного безопаснее и дешевле, чем пропустить развивающийся диабет (`FN`). Поэтому для данной задачи **разумнее выбрать порог `t = 0.3` (или `0.35`)**.

---
## Блок 4. Работа с дисбалансом классов (`class_weight='balanced'`)

In [ ]:
# 4.1 Обучение LogisticRegression(class_weight='balanced') и сравнение со стандартной моделью
logreg_bal = LogisticRegression(class_weight="balanced", random_state=RANDOM_STATE, max_iter=1000)
logreg_bal.fit(X_train_scaled, y_train)

y_pred_bal = logreg_bal.predict(X_test_scaled)
y_proba_bal = logreg_bal.predict_proba(X_test_scaled)[:, 1]

bal_metrics, cm_bal = evaluate_predictions(
    y_test, y_pred_bal, y_proba_bal, model_name="LogisticRegression (class_weight='balanced', t=0.5)"
)

comparison_all_df = pd.DataFrame([
    baseline_metrics,
    logreg_05_metrics,
    threshold_rows[0],  # t = 0.3
    threshold_rows[2],  # t = 0.7
    bal_metrics
])
display(comparison_all_df)

In [ ]:
# 4.2 Визуальное сравнение матриц ошибок: стандартная модель (t=0.5) vs class_weight='balanced' (t=0.5)
fig, axes = plt.subplots(1, 2, figsize=(11, 4.4))

sns.heatmap(
    cm_05, annot=True, fmt="d", cmap="Blues", cbar=False,
    xticklabels=["Пред. 0", "Пред. 1"], yticklabels=["Ист. 0", "Ист. 1"], ax=axes[0]
)
axes[0].set_title(f"Стандартная LogReg (t=0.5)\nRecall={logreg_05_metrics['Recall']}, FN={logreg_05_metrics['FN']}, FP={logreg_05_metrics['FP']}")

sns.heatmap(
    cm_bal, annot=True, fmt="d", cmap="Oranges", cbar=False,
    xticklabels=["Пред. 0", "Пред. 1"], yticklabels=["Ист. 0", "Ист. 1"], ax=axes[1]
)
axes[1].set_title(f"LogReg class_weight='balanced' (t=0.5)\nRecall={bal_metrics['Recall']}, FN={bal_metrics['FN']}, FP={bal_metrics['FP']}")

plt.tight_layout()
plt.show()

**Анализ `class_weight='balanced'` и почему высокий `Accuracy` не гарантирует полезность модели:**
* **Какие ошибки уменьшились, а какие выросли:** при включении `class_weight='balanced'` штраф за ошибку на редком классе `1` увеличивается пропорционально отношению частот классов (`N / (2 * N_c)`). В результате **уменьшилось число пропусков болезни `FN`** (вырос `Recall` и `F1`), но **выросло число ложных тревог `FP`** (снизился `Precision` и немного снизился или остался на том же уровне `Accuracy`). Эффект от `class_weight='balanced'` при стандартном пороге `0.5` качественно аналогичен снижению порога классификации у обычной модели до `~0.35`.
* **Почему высокий `Accuracy` сам по себе не означает, что модель полезна:** `Accuracy` считает общую долю угаданных ответов без учёта того, на каком классе модель ошибается. В выборке с дисбалансом (`65.1%` здоровых против `34.9%` больных) модель может показывать высокий `Accuracy` за счёт правильной классификации большинства здоровых людей (`TN`), одновременно пропуская значительную долю реально больных пациентов (`FN`). Более того, при переходе к `class_weight='balanced'` `Accuracy` может даже слегка снизиться из-за роста `FP`, хотя для врача модель становится гораздо полезнее за счёт высокого `Recall`.

---
## Блок 5. Итоговые выводы

### Ответы на ключевые вопросы
1. **Разница между вероятностью, порогом и итоговым классом:**
   * **Линейный скор (`z = w·x + b`) и вероятность (`p = σ(z)`)** — это внутренняя оценка уверенности модели (число от `0` до `1`, возвращаемое `predict_proba`), которая показывает, насколько объект похож на класс `1`. Сама модель при изменении порога не переобучается, и её вероятности `p` не меняются.
   * **Порог (`threshold t`)** — это внешнее правило принятия решения, выбранное человеком под конкретную бизнес- или медицинскую цену ошибки.
   * **Итоговый класс (`0` или `1`)** — результат сравнения вероятности с порогом (`p >= t -> 1`, иначе `0`).
2. **Какая ошибка в этой задаче опаснее (`FP` или `FN`):**
   * В задаче первичной диагностики диабета **намного опаснее ошибка `FN` (False Negative — ложноотрицательное срабатывание)**, когда больному пациенту модель ставит класс `0` («здоров»). Пропуск заболевания ведёт к отсутствию лечения и тяжёлым осложнениям. Ошибка `FP` (False Positive — ложная тревога) означает лишь направление здорового человека на дополнительный уточняющий анализ.
3. **Какую метрику разумнее считать основной:**
   * Основной метрикой при фиксированном пороге следует считать **`Recall` (полноту)** в связке с **`F1-мерой`** (чтобы контролировать разумный уровень `Precision` и не отправлять на дообследование 100% пациентов), а для оценки разделяющей способности модели в целом по всем порогам — **`PR-AUC` (`Average Precision`)** и **`ROC-AUC`**.

---

### 5 выводов по результатам работы
1. **Скрытые пропуски и подготовка данных:** в датасете `Pima Indians Diabetes` отсутствуют явные `NaN`, однако в физиологических признаках (`Insulin`, `SkinThickness`, `BloodPressure`, `BMI`, `Glucose`) пропуски закодированы нулями (до `48.7%` в `Insulin`). Замена таких нулей на медиану обучающей выборки и стандартизация признаков (`StandardScaler` без утечки из `test`) критически важны для корректной работы линейного классификатора.
2. **Непригодность `Accuracy` как единственной метрики:** константный baseline (`DummyClassifier`), предсказывающий всем пациентам класс `0`, достигает `Accuracy = 64.9%`, но имеет `Recall = 0.0` и `F1 = 0.0`, не выявляя ни одного случая диабета.
3. **Качество базовой логистической регрессии:** линейная модель с сигмоидой при стандартном пороге `0.5` существенно превосходит baseline по всем содержательным метрикам (`ROC-AUC > 0.80`, высокая разделяющая способность по `PR-AUC`), а наибольший положительный вклад в вероятность диабета вносят уровень глюкозы (`Glucose`), индекс массы тела (`BMI`) и число беременностей (`Pregnancies`) / наследственный фактор (`DiabetesPedigreeFunction`).
4. **Управление компромиссом `Precision–Recall` через порог:** снижение порога классификации с `0.5` до `0.3` позволяет значительно сократить число пропущенных диагнозов (`FN`) и поднять `Recall`, тогда как повышение порога до `0.7` делает модель чрезмерно консервативной и непригодной для медицинского скрининга.
5. **Учёт дисбаланса через `class_weight='balanced'`:** перевзвешивание функции потерь (`class_weight='balanced'`) смещает разделяющую гиперплоскость в пользу меньшего класса `1`, повышая полноту (`Recall`) и `F1` при стандартном пороге `0.5` ценой умеренного роста `FP`, что делает модель сразу ориентированной на минимизацию опасных ошибок второго рода (`FN`).